# Bitacora comparativa de agentes de movilidad

Este notebook ejecuta y documenta la Parte 3 del trabajo practico: la comparacion entre un agente reactivo simple y un agente reactivo basado en modelo.

La decision de cada hora usa solamente la percepcion disponible hasta esa hora. El archivo `resultado_h_mas_1.csv` queda fuera del flujo de decision porque contiene informacion futura.

## Preparacion y causalidad temporal

La logica de los agentes vive en `plantilla_agentes_movilidad.py` y el procesamiento de la secuencia en `bitacora_agentes.py`. Este notebook solamente carga el escenario, ejecuta esa logica y muestra sus resultados.

El agente simple decide con la percepcion actual. El agente basado en modelo actualiza una memoria interna, especialmente la racha de horas con presion alta, y luego decide a partir de ese estado.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Permite ejecutar el notebook desde cualquier carpeta de trabajo de VS Code.
CARPETA_NOTEBOOK = Path.cwd()
if (CARPETA_NOTEBOOK / 'bitacora_agentes.py').exists():
    CARPETA_MOVILIDAD = CARPETA_NOTEBOOK
else:
    CARPETA_MOVILIDAD = CARPETA_NOTEBOOK / 'MOVILIDAD'

if str(CARPETA_MOVILIDAD) not in sys.path:
    sys.path.insert(0, str(CARPETA_MOVILIDAD))

from bitacora_agentes import procesar_secuencia
from plantilla_agentes_movilidad import (
    actualizar_estado,
    crear_estado_inicial,
    decidir_reactivo_modelo,
    decidir_reactivo_simple,
)

In [ ]:
ruta_percepciones = CARPETA_MOVILIDAD / 'escenario_agente' / 'percepciones.csv'
percepciones = pd.read_csv(ruta_percepciones)

print(f'Archivo de entrada: {ruta_percepciones}')
display(percepciones)

## Ejecucion de la bitacora

Cada fila conserva la percepcion observada, la racha persistente del agente basado en modelo, las acciones de ambos agentes y el motivo de cada decision.

In [ ]:
bitacora = procesar_secuencia(percepciones)
display(bitacora)

ruta_salida = CARPETA_MOVILIDAD / 'bitacora_agentes.csv'
bitacora.to_csv(ruta_salida, index=False)
print(f'Bitacora guardada en: {ruta_salida}')

## Lectura de resultados

- En la primera hora con presion alta, el agente simple recomienda refuerzo inmediatamente.
- El agente basado en modelo espera dos horas consecutivas con presion alta.
- Desde la segunda hora consecutiva, ambos agentes recomiendan refuerzo.
- Si la percepcion es invalida, ambos agentes deben abstenerse.

In [ ]:
resumen = bitacora[[
    'hora', 'presion', 'racha_presion_alta',
    'accion_simple', 'accion_modelo'
]]
display(resumen)

display(bitacora[['hora', 'motivo_simple', 'motivo_modelo']])

## Pruebas obligatorias y dependencia historica

Las pruebas siguientes cubren presion baja, primera hora de presion alta, segunda hora consecutiva y dos historias diferentes que terminan en la misma percepcion.

In [ ]:
def percepcion(hora, presion):
    return {'hora': hora, 'presion': presion, 'capacidad_x': 20}

# Caso 1: presion baja.
accion_simple, _ = decidir_reactivo_simple(percepcion(6, 0.50))
estado = actualizar_estado(crear_estado_inicial(), percepcion(6, 0.50))
accion_modelo, _ = decidir_reactivo_modelo(estado)
assert accion_simple == 'NO_REFORZAR'
assert accion_modelo == 'NO_REFORZAR'

# Caso 2: primera hora con presion alta.
accion_simple, _ = decidir_reactivo_simple(percepcion(6, 1.10))
estado = actualizar_estado(crear_estado_inicial(), percepcion(6, 1.10))
accion_modelo, _ = decidir_reactivo_modelo(estado)
assert accion_simple == 'RECOMENDAR_REFUERZO'
assert accion_modelo == 'NO_REFORZAR'

# Caso 3: segunda hora consecutiva con presion alta.
estado = actualizar_estado(estado, percepcion(7, 1.20))
accion_modelo, _ = decidir_reactivo_modelo(estado)
assert accion_modelo == 'RECOMENDAR_REFUERZO'

# Caso decisivo: misma percepcion final, historias distintas.
final = percepcion(8, 1.10)
estado_sin_historia = actualizar_estado(crear_estado_inicial(), final)
estado_con_historia = actualizar_estado(
    actualizar_estado(crear_estado_inicial(), percepcion(7, 1.10)),
    final,
)
simple_a, _ = decidir_reactivo_simple(final)
simple_b, _ = decidir_reactivo_simple(final)
modelo_a, _ = decidir_reactivo_modelo(estado_sin_historia)
modelo_b, _ = decidir_reactivo_modelo(estado_con_historia)
assert simple_a == simple_b == 'RECOMENDAR_REFUERZO'
assert modelo_a != modelo_b

# Causalidad temporal: se carga la evaluacion real de h+1 solo para auditarla.
ruta_evaluacion = CARPETA_MOVILIDAD / 'escenario_agente' / 'resultado_h_mas_1.csv'
evaluacion_futura = pd.read_csv(ruta_evaluacion)
percepcion_h = percepciones.sort_values('hora').iloc[-1].to_dict()
fila_h_mas_1 = evaluacion_futura.sort_values('hora').iloc[0].to_dict()
assert fila_h_mas_1['hora'] == percepcion_h['hora'] + 1

# El dato real de h+1 se observa despues, pero nunca se entrega al agente.
estado_h = actualizar_estado(crear_estado_inicial(), percepcion_h)
simple_h, _ = decidir_reactivo_simple(percepcion_h)
modelo_h, _ = decidir_reactivo_modelo(estado_h)
assert 'resultado_h_mas_1.csv' not in str(ruta_percepciones)
assert 'presion' in fila_h_mas_1

print('Pruebas aprobadas: la evaluacion real de h+1 se carga solo despues de decidir y no se usa como percepcion.')

## Respuestas de la Parte 3

1. Ambos agentes producen la misma accion con presion baja, con presion alta persistente durante al menos dos horas consecutivas, o ante datos invalidos.
2. Reaccionan diferente durante la primera hora de presion alta: el agente simple recomienda refuerzo y el agente basado en modelo todavia no.
3. El segundo es basado en modelo porque conserva un estado interno que resume la historia (`racha_presion_alta`). No necesita planificar para ser un agente basado en modelo.
4. `tasa_otras_simulada` es una proporcion sintetica y aleatoria de viajes asignada a otras empresas. No permite inferir la cantidad real de taxis competidores ni una cuota de mercado real.
5. `resultado_h_mas_1.csv` no puede formar parte de la percepcion porque contiene datos futuros. Usarlo produciria fuga de informacion y romperia la causalidad temporal.

## PEAS y limitaciones

| Elemento | Descripcion |
|---|---|
| Performance | Recomendaciones coherentes, sin fuga temporal, con abstencion ante datos invalidos y trazabilidad. |
| Environment | Secuencia simulada de zona y hora, demanda TLC transformada, flota sintetica de X, otras empresas sinteticas y responsable humano. |
| Actuators | Mensajes `NO_REFORZAR`, `RECOMENDAR_REFUERZO` y `ABSTENERSE`. |
| Sensors | Lectura logica de `percepciones.csv`; no es un sensor conectado en tiempo real. |

Las observaciones provienen de viajes Yellow Taxi realizados y reportados, no de toda la demanda ni de solicitudes no atendidas. X y las otras empresas son ficticias, la relacion entre flota y participacion externa es una hipotesis didactica y una unidad de capacidad por taxi-hora es una simplificacion. `RECOMENDAR_REFUERZO` es una recomendacion para revision humana, no una orden de traslado.